In [1]:
from pathlib import Path
import src.utils.features as features
from src.utils.notebook_ploting import distribution_explorer

import pandas as pd

DATA_DIR = Path("../datasets/final/ml")

AVAILABLE_DATASETS = ["cicids2017", "unsw_nb15", "iot23"]
DATASET_NAME = AVAILABLE_DATASETS[2]

df = pd.read_parquet(
    DATA_DIR / f"{DATASET_NAME}_train.parquet"
)


In [2]:
#essas flags apareceram com variação nula em pelo menos 1 das bases
# dst2src_urg_packets apareceu sem variação nas 3 bases. por esse motivo, já foi desconsiderado.

tcp_flag_features = [
    "bidirectional_cwr_packets",
    "bidirectional_ece_packets",
    "bidirectional_urg_packets",
    "src2dst_cwr_packets",
    "src2dst_ece_packets",
    "src2dst_urg_packets",
    "dst2src_cwr_packets",
    "dst2src_ece_packets",
    "dst2src_rst_packets",
]


In [3]:
summary = df[tcp_flag_features].agg(
    ["min", "max", "mean", "std"]
).T

summary["non_zero"] = (df[tcp_flag_features] > 0).sum()
summary["non_zero_pct"] = (
    summary["non_zero"] / len(df) * 100
).round(4)

display(summary.sort_values("non_zero_pct"))
# resultados por dataset
# cicids2017:
# quase todas as flags são extremante raras, abaixo de 0.0204 % de ocorrencia.
# apenas dst2src_rst_packets apresentou 9.6314 %

# unsw
# as 9 flags são constantes em 0, não acrescentam informação

,min,max,mean,std,non_zero,non_zero_pct
bidirectional_ece_packets,0.0,0.0,0.000000,0.000000,0,0.0000
dst2src_ece_packets,0.0,0.0,0.000000,0.000000,0,0.0000
dst2src_cwr_packets,0.0,0.0,0.000000,0.000000,0,0.0000
src2dst_ece_packets,0.0,0.0,0.000000,0.000000,0,0.0000
bidirectional_cwr_packets,0.0,81.0,0.036002,1.261176,344,0.1461
src2dst_cwr_packets,0.0,81.0,0.036002,1.261176,344,0.1461
bidirectional_urg_packets,0.0,81.0,0.036006,1.261178,345,0.1465
src2dst_urg_packets,0.0,81.0,0.036006,1.261178,345,0.1465
dst2src_rst_packets,0.0,4.0,0.002446,0.058452,475,0.2017


In [4]:
occurrence_by_label = (
    df[tcp_flag_features]
    .gt(0)
    .groupby(df["label_binary"])
    .mean()
    .T
    .mul(100)
)

display(occurrence_by_label)

label_binary,ATTACK,BENIGN
bidirectional_cwr_packets,0.730392,0.000000
bidirectional_ece_packets,0.000000,0.000000
bidirectional_urg_packets,0.732515,0.000000
src2dst_cwr_packets,0.730392,0.000000
src2dst_ece_packets,0.000000,0.000000
src2dst_urg_packets,0.732515,0.000000
dst2src_cwr_packets,0.000000,0.000000
dst2src_ece_packets,0.000000,0.000000
dst2src_rst_packets,0.076436,0.233025


In [5]:
numerical, categorical= features.split_features(df)
# já desconsidera features inúteis

In [6]:
display(numerical)

['bidirectional_duration_ms',
 'bidirectional_packets',
 'bidirectional_bytes',
 'bidirectional_min_ps',
 'bidirectional_mean_ps',
 'bidirectional_stddev_ps',
 'bidirectional_max_ps',
 'bidirectional_min_piat_ms',
 'bidirectional_mean_piat_ms',
 'bidirectional_stddev_piat_ms',
 'bidirectional_max_piat_ms',
 'bidirectional_syn_packets',
 'bidirectional_ack_packets',
 'bidirectional_psh_packets',
 'bidirectional_rst_packets',
 'bidirectional_fin_packets']

In [7]:
display(categorical)

['src_port', 'dst_port', 'protocol', 'ip_version']

In [8]:
distribution_explorer(df[numerical + ['label_binary']])
#    ("bidirectional_packets", "src2dst_packets"),
# 3 histrogramas para facilitar a viosualização de src-> dst; dst -> src; bidirecional

In [9]:
distribution_explorer(df[numerical + ['label_binary']])

In [10]:
distribution_explorer(df[numerical + ['label_binary']])

In [11]:
df["bidirectional_duration_ms"].describe()

count    235490.000000
mean       5410.304633
std        4274.632505
min           0.000000
25%           0.000000
50%        7187.000000
75%        7215.000000
max      119983.000000
Name: bidirectional_duration_ms, dtype: float64